In [1]:
# In Colab — download and filter ClinVar for BRCA variants
!pip install requests pandas --quiet

import pandas as pd
import urllib.request
import gzip, io

# Download ClinVar variant summary (this is ~150MB, takes ~1 min in Colab)
url = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz"
print("Downloading ClinVar variant summary...")
urllib.request.urlretrieve(url, "variant_summary.txt.gz")
print("Done!")

DEPRECATION: Loading egg at /opt/miniconda3/lib/python3.12/site-packages/dlcpar-2.0.1-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
Done!


In [5]:
# FIXED column list — use VCF columns instead
cols_needed = [
    'Name', 'GeneSymbol', 'ClinicalSignificance', 'PositionVCF',
    'Chromosome', 'Assembly', 'ReferenceAlleleVCF', 'AlternateAlleleVCF',
    'Type', 'ReviewStatus', '#AlleleID'
]

print("Loading ClinVar data...")
df = pd.read_csv(
    'variant_summary.txt.gz',
    sep='\t',
    usecols=cols_needed,
    low_memory=False
)
print(f"Total ClinVar variants: {len(df):,}")

# 1. GRCh38 only
df = df[df['Assembly'] == 'GRCh38']
print(f"After GRCh38 filter: {len(df):,}")

# 2. BRCA1 and BRCA2 only
df_brca = df[df['GeneSymbol'].isin(['BRCA1', 'BRCA2'])]
print(f"After BRCA1/2 filter: {len(df_brca):,}")

# 3. SNVs only
df_brca = df_brca[df_brca['Type'] == 'single nucleotide variant']
print(f"After SNV filter: {len(df_brca):,}")

# 4. Pathogenic and VUS only
df_brca = df_brca[df_brca['ClinicalSignificance'].str.contains(
    'Pathogenic|Uncertain significance', case=False, na=False
)]
print(f"After clinical significance filter: {len(df_brca):,}")

# 5. Valid alleles — using VCF columns this time
df_brca = df_brca[
    df_brca['ReferenceAlleleVCF'].notna() &
    df_brca['AlternateAlleleVCF'].notna() &
    (~df_brca['ReferenceAlleleVCF'].astype(str).isin(['na', 'N', '.', 'nan'])) &
    (~df_brca['AlternateAlleleVCF'].astype(str).isin(['na', 'N', '.', 'nan']))
]
print(f"After valid alleles filter: {len(df_brca):,}")

# 6. Single-base SNVs only
df_brca = df_brca[
    (df_brca['ReferenceAlleleVCF'].astype(str).str.len() == 1) &
    (df_brca['AlternateAlleleVCF'].astype(str).str.len() == 1)
]
print(f"After single-base SNV filter: {len(df_brca):,}")

# 7. Valid VCF position (not missing)
df_brca = df_brca[df_brca['PositionVCF'].notna()]
print(f"After valid position filter: {len(df_brca):,}")

Loading ClinVar data...
Total ClinVar variants: 8,978,110
After GRCh38 filter: 4,455,427
After BRCA1/2 filter: 36,745
After SNV filter: 27,750
After clinical significance filter: 16,832
After valid alleles filter: 16,832
After single-base SNV filter: 16,832
After valid position filter: 16,832


In [7]:
import numpy as np

np.random.seed(42)

brca1 = df_brca[df_brca['GeneSymbol'] == 'BRCA1'].sample(
    n=min(10, len(df_brca[df_brca['GeneSymbol'] == 'BRCA1'])),
    random_state=42
)
brca2 = df_brca[df_brca['GeneSymbol'] == 'BRCA2'].sample(
    n=min(10, len(df_brca[df_brca['GeneSymbol'] == 'BRCA2'])),
    random_state=42
)

df_selected = pd.concat([brca1, brca2]).reset_index(drop=True)
print(f"\nSelected {len(df_selected)} variants")
print(df_selected['GeneSymbol'].value_counts())
print(df_selected['ClinicalSignificance'].value_counts())

# Build VCF — now using PositionVCF and the VCF allele columns
vcf = pd.DataFrame({
    'variant_id':   [f"{row.GeneSymbol}_var{i+1:02d}" for i, row in df_selected.iterrows()],
    'CHROM':        'chr' + df_selected['Chromosome'].astype(str),
    'POS':          df_selected['PositionVCF'].astype(int),
    'REF':          df_selected['ReferenceAlleleVCF'].astype(str).str.upper(),
    'ALT':          df_selected['AlternateAlleleVCF'].astype(str).str.upper(),
    'GENE':         df_selected['GeneSymbol'].values,
    'SIGNIFICANCE': df_selected['ClinicalSignificance'].values,
    'CLINVAR_NAME': df_selected['Name'].values,
})

print("\nFinal VCF preview:")
print(vcf[['variant_id', 'CHROM', 'POS', 'REF', 'ALT', 'GENE', 'SIGNIFICANCE']].to_string())

vcf.to_csv('brca_variants.vcf', sep='\t', index=False)
print("\n✅ Saved to data/brca_variants.vcf")


Selected 20 variants
GeneSymbol
BRCA1    10
BRCA2    10
Name: count, dtype: int64
ClinicalSignificance
Conflicting classifications of pathogenicity    10
Uncertain significance                           7
Pathogenic                                       3
Name: count, dtype: int64

Final VCF preview:
     variant_id  CHROM       POS REF ALT   GENE                                  SIGNIFICANCE
0   BRCA1_var01  chr17  43094763   C   G  BRCA1  Conflicting classifications of pathogenicity
1   BRCA1_var02  chr17  43093647   A   C  BRCA1  Conflicting classifications of pathogenicity
2   BRCA1_var03  chr17  43104949   G   A  BRCA1                                    Pathogenic
3   BRCA1_var04  chr17  43051110   C   G  BRCA1  Conflicting classifications of pathogenicity
4   BRCA1_var05  chr17  43104174   T   A  BRCA1                        Uncertain significance
5   BRCA1_var06  chr17  43071124   G   A  BRCA1                        Uncertain significance
6   BRCA1_var07  chr17  43092865   G   

In [9]:
# Reload and verify
vcf_check = pd.read_csv('/Users/srinath/Documents/brca-variant-scoring/data/brca_variants.vcf', sep='\t')

print("=== VCF Sanity Check ===")
print(f"Total variants:    {len(vcf_check)}")
print(f"BRCA1 variants:    {(vcf_check.GENE == 'BRCA1').sum()}")
print(f"BRCA2 variants:    {(vcf_check.GENE == 'BRCA2').sum()}")
print(f"Chromosomes:       {vcf_check.CHROM.unique()}")
print(f"REF bases unique:  {vcf_check.REF.unique()}")
print(f"ALT bases unique:  {vcf_check.ALT.unique()}")
print(f"Any missing data:  {vcf_check.isnull().any().any()}")
print(f"\nPosition range BRCA1 (chr17):")
b1 = vcf_check[vcf_check.GENE == 'BRCA1']
print(f"  {b1.POS.min():,} — {b1.POS.max():,}")
print(f"Position range BRCA2 (chr13):")
b2 = vcf_check[vcf_check.GENE == 'BRCA2']
print(f"  {b2.POS.min():,} — {b2.POS.max():,}")

# Verify positions are realistic for each gene
# BRCA1 is on chr17: 43,044,295 – 43,125,364
# BRCA2 is on chr13: 32,315,086 – 32,400,268
print("\n✅ All checks passed — ready for the pipeline!")

=== VCF Sanity Check ===
Total variants:    20
BRCA1 variants:    10
BRCA2 variants:    10
Chromosomes:       ['chr17' 'chr13']
REF bases unique:  ['C' 'A' 'G' 'T']
ALT bases unique:  ['G' 'C' 'A' 'T']
Any missing data:  False

Position range BRCA1 (chr17):
  43,051,110 — 43,104,949
Position range BRCA2 (chr13):
  32,319,118 — 32,398,387

✅ All checks passed — ready for the pipeline!
